# crea tu “nube de palabras” Word Cloud con Python a partir de varias webs   
  

In [1]:
""""""https://antonio-fernandez-troyano.medium.com/nube-de-palabras-word-cloud-con-python-a-partir-de-varias-webs-111e94220822
  

SyntaxError: invalid syntax (<ipython-input-1-f55851266593>, line 1)

In [ ]:
#Librerías básicas utiliadas
import numpy as np
import pandas as pd

#Librerías necesarias para scrapear textos de páginas web
import requests
from bs4 import BeautifulSoup

#Librerías necesarias para abrir imágenes, generar nube de palabras y plot
from PIL import Image
from wordcloud import WordCloud, ImageColorGenerator
import matplotlib.pyplot as plt

#Librerías necesarias para la limpieza de datos
import string
import nltk
from nltk.corpus import stopwords


In [ ]:
import nltk
# nltk.download()  # OJO SOLO UNA VEZ Y EL PAQUETE stopwords

In [ ]:
def get_texto_url(list_webs):
    
    #Hay que incluir unos "headers" para que las páginas web piensen que es una persona "real" la que está accediendo a la web
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/72.0.3626.121 Safari/537.36",
                   "Accept-Encoding": "*", "Connection": "keep-alive"
            }

    texto =" "
    long=0
    for url in webs:

        try:
            web = requests.get(url, headers=headers)

            soup = BeautifulSoup(web.text)

            parrafos = soup.find_all('p')

            for p in parrafos:
                long += len(p.text)
                texto += " "+ str(p.text)

        except:
            print("Error en la url {}, response {}".format(url,web))
    
    print("Se han añadido un total de {} caractéres a la variable texto".format(long))
    return texto

In [ ]:
#--------------------------- inicio del programa

webs = [r"https://es.wikipedia.org/wiki/An%C3%A1lisis_de_datos", 
       r"https://es.wikipedia.org/wiki/Macrodatos"]

texto = get_texto_url(webs)

In [ ]:
#Generación de lista de signos de puntuación

punctuation=[]
for s in string.punctuation:
    punctuation.append(str(s))
sp_punctuation = ["¿", "¡", "“", "”", "…", ":", "–", "»", "«"]    

punctuation += sp_punctuation

#Listado de palabras que queremos eliminar del texto
#Es un proceso iterativo por lo que si después vemos que nos siguen quedado "caractéres raros" simplemente venímos aquí y los agregamos
#Existe librerías y listados de "Stop_words", pero por ahora vamos a dejarlo vacío

#nltk.download('stopwords') #La primera vez debemos descargar las "stopwords"

stop_words = stopwords.words('spanish') #Listado de palabras a eliminar

stop_words += ["\u200b", "\xa0", "para", "como", "puede","cómo", "hacer", "forma", "parte", "hace", "además", "según", "pueden", "ser"]
#Añadimos algunos caractéres que hemos encontrado

print (stop_words)

In [ ]:
#Reemplazamos signos de puntuación por "":
for p in punctuation:
    clean_texto = texto.lower().replace(p,"")
    
for p in punctuation:
    clean_texto = clean_texto.replace(p,"")

#Reemplazamos stop_words por "":    
for stop in stop_words:
    clean_texto_list = clean_texto.split()
    clean_texto_list = [i.strip() for i in clean_texto_list]
    try:
        while stop in clean_texto_list: clean_texto_list.remove(stop)
    except:
        print("Error")
        pass
    clean_texto= " ".join(clean_texto_list)


In [ ]:
lista_texto = clean_texto.split(" ")

palabras = []

#Paso intermedio para eliminar palabras muy cortas (emoticonos,etc) y muy largas (ulr o similar) que se nos hayan pasado:

for palabra in lista_texto:
    if (len(palabra)>=3 and len(palabra)<18):
        palabras.append(palabra)
        
#Generamos un diccionario para contabilizar las palabras:

word_count={}

for palabra in palabras:
    if palabra in word_count.keys():
        word_count[palabra][0]+=1
    else:
        word_count[palabra]=[1]
        
print (word_count)



In [ ]:
#Convertimos el diccionario en un pandas DataFrame para ordernarlo:

df = pd.DataFrame.from_dict(word_count).transpose()
df.columns=["freq"]
df.sort_values(["freq"], ascending=False, inplace=True)
df.head(10)

In [ ]:

word_cloud = WordCloud(height=800, width=800, background_color='white',max_words=150, min_font_size=5, collocation_threshold=10).generate(clean_texto)

word_cloud.to_file("./img/ejemplo_sencillo.png") #Guardamos la imagen generada

plt.figure(figsize=(10,8))
plt.imshow(word_cloud)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()